## 1. Kurulum ve Bağımlılıklar

In [ ]:
# GPU kontrolü
!nvidia-smi

In [ ]:
# Repository'yi klonla
!git clone https://github.com/Aliekinozcetin/DiffuVQA.git
%cd DiffuVQA

# Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

# OPSIYONEL 1: Dataset'i kopyalamak yerine symlink kullan (ÇOOK HIZLI!)
# Sadece link oluşturur, kopyalama yapmaz
!ln -s /content/drive/MyDrive/SLAKE_dataset ./datasets
print("✅ Dataset symlink ile bağlandı (kopyalama YOK, anında!)")

# OPSIYONEL 2: Eğer mutlaka kopyalama gerekiyorsa (daha yavaş ama stable)
# Önce kontrol et, varsa kopyalama
# import os
# if not os.path.exists('./datasets'):
#     print("📦 Dataset kopyalanıyor... (Bu birkaç dakika sürebilir)")
#     !cp -r /content/drive/MyDrive/SLAKE_dataset ./datasets
#     print("✅ Dataset kopyalandı")
# else:
#     print("✅ Dataset zaten mevcut")

# OPSIYONEL 3: Eğer dataset ZIP ise (EN HIZLI kopyalama yöntemi)
# !cp /content/drive/MyDrive/SLAKE_dataset.zip ./
# !unzip -q SLAKE_dataset.zip -d ./datasets
# print("✅ Dataset ZIP'ten açıldı")

In [ ]:
# Dataset yapısını kontrol et
!echo "📂 Dataset yapısı:"
!ls -lh datasets/
!echo "\n📊 Train set örneği:"
!head -n 1 datasets/train.jsonl

In [ ]:
# Bağımlılıkları yükle
# Colab PyTorch ile geliyor, o yüzden requirements_colab.txt kullanıyoruz
!pip install --upgrade pip
!pip install -r requirements_colab.txt
!pip install pandas openpyxl  # Excel support için
!python -m spacy download en_core_web_sm

print("\n✅ Kurulum tamamlandı!")

In [ ]:
# Gerekli kütüphaneleri içe aktar
import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from datetime import datetime
from collections import defaultdict
import glob

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Model Eğitimi (PubMedBERT)

In [ ]:
# Eğitim konfigürasyonu
training_config = {
    "vocab": "pubmedbert",  # PubMedBERT tokenizer kullan
    "init_pretrained": "pubmedbert",  # PubMedBERT embeddings kullan
    "batch_size": 4,
    "learning_rate": 0.0001,
    "num_steps": 100,
    "diffusion_steps": 100,
    "seq_len": 128,
    "hidden_size": 768,  # PubMedBERT hidden dimension
    "image_resolution": 224,
    "seed": 42,
    "checkpoint_path": "./checkpoints/pubmedbert_slake"
}

# Checkpoint dizinini oluştur
os.makedirs(training_config["checkpoint_path"], exist_ok=True)

print("Eğitim Konfigürasyonu:")
for k, v in training_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Model eğitimini başlat
!python train.py \
    --vocab pubmedbert \
    --init_pretrained pubmedbert \
    --batch_size 4 \
    --lr 0.0001 \
    --diffusion_steps 100 \
    --seq_len 128 \
    --hidden_t_dim 768 \
    --checkpoint_path ./checkpoints/pubmedbert_slake \
    --seed 42 \
    --num_steps 1000

## 3. Model Örnekleme (Inference)

In [ ]:
# Test seti üzerinde örnekleme yap
checkpoint_file = "./checkpoints/pubmedbert_slake/ema_0.9999_100000.pt"  # Checkpoint dosyasını güncelle
sample_output = "./samples/pubmedbert_slake_samples.jsonl"

!python sample_vqa_GPU.py \
    --model_path {checkpoint_file} \
    --vocab pubmedbert \
    --init_pretrained pubmedbert \
    --batch_size 4 \
    --num_samples 10 \
    --top_p 0.9 \
    --seed 123 \
    --output_file {sample_output}

print(f"\n✅ Örnekleme tamamlandı: {sample_output}")

## 4. Model Değerlendirme ve CSV Export

In [ ]:
# Değerlendirme fonksiyonları
from torchmetrics.text.rouge import ROUGEScore
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# NLTK veri setlerini indir
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

rougeScore = ROUGEScore()

def exact_match(prediction, reference):
    """Exact match accuracy"""
    return 1.0 if prediction.strip().lower() == reference.strip().lower() else 0.0

def get_bleu(recover, reference, n=1):
    """BLEU-n score"""
    weights = tuple((1.0 / n for _ in range(n)))
    return sentence_bleu([reference.split()], recover.split(), 
                        weights=weights, 
                        smoothing_function=SmoothingFunction().method4)

def calculate_meteor(recover, reference):
    """METEOR score"""
    try:
        return meteor_score([word_tokenize(reference)], word_tokenize(recover))
    except:
        return 0.0

def cider_score(candidates, references):
    """CIDEr-like score using TF-IDF cosine similarity"""
    vectorizer = TfidfVectorizer()
    all_sentences = candidates + references
    
    try:
        tfidf_matrix = vectorizer.fit_transform(all_sentences).toarray()
        candidates_tfidf = tfidf_matrix[:len(candidates)]
        references_tfidf = tfidf_matrix[len(candidates):]
        
        cider_scores = []
        for i, candidate_tfidf in enumerate(candidates_tfidf):
            ref_tfidf = references_tfidf[i]
            similarity = cosine_similarity(candidate_tfidf.reshape(1, -1), 
                                          ref_tfidf.reshape(1, -1))
            cider_scores.append(similarity[0][0])
        
        return np.mean(cider_scores)
    except:
        return 0.0

def rouge_l_score(prediction, reference):
    """ROUGE-L score"""
    try:
        scores = rougeScore([prediction], [reference])
        return scores['rougeL_fmeasure'].item()
    except:
        return 0.0

print("✅ Değerlendirme fonksiyonları yüklendi")

In [ ]:
def evaluate_and_export_csv(sample_files, output_csv="evaluation_results.csv"):
    """
    Örneklenmiş model çıktılarını değerlendir ve CSV'ye kaydet
    
    Args:
        sample_files: JSONL formatında örnek dosyaları (liste veya tek dosya)
        output_csv: Çıktı CSV dosya yolu
    """
    if isinstance(sample_files, str):
        sample_files = [sample_files]
    
    all_results = []
    
    for sample_file in sample_files:
        print(f"\n📊 Değerlendiriliyor: {sample_file}")
        
        # JSONL dosyasını oku
        samples = []
        with open(sample_file, 'r', encoding='utf-8') as f:
            for line in f:
                samples.append(json.loads(line))
        
        # Metrikler için listeler
        exact_matches = []
        bleu1_scores = []
        rougeL_scores = []
        meteor_scores = []
        
        predictions = []
        references = []
        
        # Accuracy için sayaçlar
        total_samples = 0
        correct_all = 0
        correct_yn = 0
        correct_oe = 0
        count_yn = 0
        count_oe = 0
        
        # Her örnek için metrikleri hesapla
        for sample in samples:
            # Farklı JSON key'lerini dene
            pred = (sample.get('recover') or 
                   sample.get('generated_answer') or 
                   sample.get('prediction') or '').strip()
            
            ref = (sample.get('reference') or 
                  sample.get('reference_answer') or 
                  sample.get('answer') or '').strip()
            
            q_type = sample.get('answer_type', 'unknown')
            
            if not pred or not ref:
                continue
            
            predictions.append(pred)
            references.append(ref)
            
            # Exact match
            em = exact_match(pred, ref)
            exact_matches.append(em)
            
            # Accuracy hesapla
            total_samples += 1
            if em == 1.0:
                correct_all += 1
                if q_type == 'CLOSED':
                    correct_yn += 1
                elif q_type == 'OPEN':
                    correct_oe += 1
            
            if q_type == 'CLOSED':
                count_yn += 1
            elif q_type == 'OPEN':
                count_oe += 1
            
            # BLEU-1
            bleu1_scores.append(get_bleu(pred, ref, n=1))
            
            # ROUGE-L
            rougeL_scores.append(rouge_l_score(pred, ref))
            
            # METEOR
            meteor_scores.append(calculate_meteor(pred, ref))
        
        # CIDEr-like score
        cider = cider_score(predictions, references) if predictions else 0.0
        
        # Ortalama metrikleri hesapla
        results = {
            'Sample File': os.path.basename(sample_file),
            'Model': 'DiffuVQA-PubMedBERT',
            'Dataset': 'SLAKE',
            'Total Samples': total_samples,
            'Evaluation Date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'Overall Accuracy': correct_all / total_samples if total_samples > 0 else 0.0,
            'Yes/No Accuracy': correct_yn / count_yn if count_yn > 0 else 0.0,
            'Open-Ended Accuracy': correct_oe / count_oe if count_oe > 0 else 0.0,
            'Exact Match': np.mean(exact_matches) if exact_matches else 0.0,
            'BLEU-1': np.mean(bleu1_scores) if bleu1_scores else 0.0,
            'ROUGE-L': np.mean(rougeL_scores) if rougeL_scores else 0.0,
            'METEOR': np.mean(meteor_scores) if meteor_scores else 0.0,
            'CIDEr': cider
        }
        
        all_results.append(results)
        
        # Sonuçları yazdır
        print(f"\n📈 Sonuçlar - {os.path.basename(sample_file)}:")
        print(f"  Total Samples: {results['Total Samples']}")
        print(f"  Overall Accuracy: {results['Overall Accuracy']:.4f}")
        print(f"  Yes/No Accuracy: {results['Yes/No Accuracy']:.4f}")
        print(f"  Open-Ended Accuracy: {results['Open-Ended Accuracy']:.4f}")
        print(f"  BLEU-1: {results['BLEU-1']:.4f}")
        print(f"  ROUGE-L: {results['ROUGE-L']:.4f}")
        print(f"  METEOR: {results['METEOR']:.4f}")
        print(f"  CIDEr: {results['CIDEr']:.4f}")
    
    # DataFrame oluştur ve CSV'ye kaydet
    df = pd.DataFrame(all_results)
    df.to_csv(output_csv, index=False, encoding='utf-8')
    
    print(f"\n✅ Sonuçlar CSV'ye kaydedildi: {output_csv}")
    
    return df

print("✅ CSV export fonksiyonu hazır")

In [ ]:
# Tüm örnek dosyalarını değerlendir
sample_folder = "./samples/"
sample_files = glob.glob(f"{sample_folder}/*.jsonl")

if not sample_files:
    print("⚠️ Örnek dosyası bulunamadı!")
else:
    print(f"📂 {len(sample_files)} örnek dosyası bulundu")
    
    # CSV çıktı dosyası
    output_csv = "./reports/pubmedbert_evaluation_results.csv"
    os.makedirs("./reports", exist_ok=True)
    
    # Değerlendirme yap ve CSV'ye kaydet
    results_df = evaluate_and_export_csv(sample_files, output_csv=output_csv)
    
    # Sonuçları görüntüle
    print("\n" + "="*80)
    print("📊 TÜM SONUÇLAR:")
    print("="*80)
    display(results_df)

## 5. Sonuçları İndir

In [ ]:
# CSV dosyasını Google Drive'a kaydet ve indir
from google.colab import files

# CSV'yi Google Drive'a kaydet
drive_output_path = '/content/drive/MyDrive/DiffuVQA_Results/'
os.makedirs(drive_output_path, exist_ok=True)

if os.path.exists(output_csv):
    # Drive'a kopyala
    import shutil
    shutil.copy(output_csv, drive_output_path)
    print(f"✅ CSV Google Drive'a kaydedildi: {drive_output_path}")
    
    # Lokal olarak da indir
    files.download(output_csv)
    print(f"✅ {output_csv} bilgisayarınıza indirildi")

# Checkpoint'leri de Drive'a yedekle (opsiyonel)
# !cp -r ./checkpoints /content/drive/MyDrive/DiffuVQA_Checkpoints/

## 6. Sonuç Analizi ve Görselleştirme

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sonuçları görselleştir
if 'results_df' in locals():
    # Metrik sütunlarını seç
    metric_cols = ['Overall Accuracy', 'Yes/No Accuracy', 'Open-Ended Accuracy', 
                   'BLEU-1', 'ROUGE-L', 'METEOR', 'CIDEr']
    
    # Bar plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    
    # Accuracy metrikleri
    accuracy_cols = ['Overall Accuracy', 'Yes/No Accuracy', 'Open-Ended Accuracy']
    if len(results_df) > 0:
        results_df[accuracy_cols].iloc[0].plot(kind='bar', ax=axes[0], color='skyblue')
        axes[0].set_title('Accuracy Metrics', fontsize=14, fontweight='bold')
        axes[0].set_ylabel('Score', fontsize=12)
        axes[0].set_ylim([0, 1])
        axes[0].grid(axis='y', alpha=0.3)
        
        # NLG metrikleri
        nlg_cols = ['BLEU-1', 'ROUGE-L', 'METEOR', 'CIDEr']
        results_df[nlg_cols].iloc[0].plot(kind='bar', ax=axes[1], color='lightcoral')
        axes[1].set_title('NLG Metrics', fontsize=14, fontweight='bold')
        axes[1].set_ylabel('Score', fontsize=12)
        axes[1].set_ylim([0, 1])
        axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('./reports/metrics_visualization.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("✅ Görselleştirme kaydedildi: ./reports/metrics_visualization.png")

## 7. Model Karşılaştırma (İsteğe Bağlı)

In [ ]:
# Farklı model çıktılarını karşılaştır
# Örneğin: BERT vs PubMedBERT

def compare_models(csv_files, labels):
    """
    Farklı modellerin CSV sonuçlarını karşılaştır
    
    Args:
        csv_files: CSV dosya yolları listesi
        labels: Model isimleri listesi
    """
    comparison_data = []
    
    for csv_file, label in zip(csv_files, labels):
        if os.path.exists(csv_file):
            df = pd.read_csv(csv_file)
            df['Model_Label'] = label
            comparison_data.append(df)
    
    if comparison_data:
        combined_df = pd.concat(comparison_data, ignore_index=True)
        
        # Karşılaştırma grafiği
        metric_cols = ['Overall Accuracy', 'BLEU-1', 'ROUGE-L', 'METEOR', 'CIDEr']
        
        fig, ax = plt.subplots(figsize=(12, 6))
        combined_df.groupby('Model_Label')[metric_cols].mean().plot(kind='bar', ax=ax)
        ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
        ax.set_ylabel('Score', fontsize=12)
        ax.set_ylim([0, 1])
        ax.legend(loc='upper right')
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig('./reports/model_comparison.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        return combined_df
    
    return None

# Örnek kullanım:
# compare_df = compare_models(
#     csv_files=['./reports/bert_results.csv', './reports/pubmedbert_results.csv'],
#     labels=['BERT-base', 'PubMedBERT']
# )

---

## Notlar

- **PubMedBERT:** Medical domain'e özgü pre-trained embeddings kullanır
- **SLAKE Dataset:** Medical VQA için kullanılır
- **CSV Export:** Tüm metrikler otomatik olarak CSV'ye kaydedilir
- **Checkpoint:** Model checkpoint'leri `./checkpoints/` dizinine kaydedilir
- **Samples:** Model çıktıları `./samples/` dizinine kaydedilir

---

**Hazırlayan:** DiffuVQA Team  
**Tarih:** 8 Aralık 2025